# RealVeritas AI - Audio Model Training (Ultra Disk-Efficient Pipeline)
Because `gdown` fails on massive datasets like ASVspoof, we will use a **Google Drive Shortcut** instead.

This ultra-efficient version is designed to bypass the Colab 'Disk Full' limit by immediately deleting zip files after extracting, and moving files instead of copying them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install librosa soundfile tqdm torchaudio

### Step 1: Smart-Extract Dataset & Manage Disk Space
This cell reads directly from your Drive shortcut, unzips everything into Colab's fast memory, deletes the heavy zip files immediately to save space, and neatly sorts every single `.wav` file.

In [ ]:
import os
import shutil
import zipfile

raw_dataset = "/content/raw_dataset"
SHORTCUT_PATH = "/content/drive/MyDrive/AudioDataset"

if not os.path.exists(SHORTCUT_PATH):
    raise Exception("Shortcut not found! Please create a shortcut to the shared folder in your My Drive and name it 'AudioDataset'.")

print("Copying files from Drive shortcut into Colab's fast memory...")
shutil.copytree(SHORTCUT_PATH, raw_dataset, dirs_exist_ok=True)

DATASET_AUDIO_PATH = "/content/dataset_audio"
authentic_dir = os.path.join(DATASET_AUDIO_PATH, "authentic")
ai_generated_dir = os.path.join(DATASET_AUDIO_PATH, "ai_generated")
os.makedirs(authentic_dir, exist_ok=True)
os.makedirs(ai_generated_dir, exist_ok=True)

print("\nUnzipping dataset archives & aggressively cleaning up disk space...")
for root, dirs, files in os.walk(raw_dataset):
    for f in files:
        if f.endswith('.zip'):
            zip_path = os.path.join(root, f)
            extract_to = os.path.join(root, f.replace('.zip', ''))
            print(f"Extracting {f}...")
            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_to)
                # DELTE ZIP TO SAVE MASSIVE DISK SPACE
                os.remove(zip_path)
            except Exception as e:
                print(f"Failed to extract {f}: {e}")

label_map = {}
print("\nScanning ASVspoof protocol files to determine audio labels...")
for root, dirs, files in os.walk(raw_dataset):
    for f in files:
        if f.endswith('.txt') or f.endswith('.trn'):
            try:
                with open(os.path.join(root, f), 'r', encoding='utf-8') as txt_file:
                    for line in txt_file:
                        line = line.strip().lower()
                        if not line: continue
                        parts = line.split()
                        
                        is_authentic = 'bonafide' in parts or 'genuine' in parts
                        is_fake = 'spoof' in parts
                        
                        if is_authentic or is_fake:
                            for p in parts:
                                if p.startswith('t_') or p.startswith('la_') or p.startswith('pa_') or p.startswith('e_') or p.startswith('d_') or '.wav' in p:
                                    fname = p.replace('.wav', '')
                                    label_map[fname] = 'authentic' if is_authentic else 'ai_generated'
            except Exception:
                pass

print(f"Found {len(label_map)} labeled files from ASV protocols.")

print("\nRouting audio files to 'authentic' or 'ai_generated' (Moving instead of copying to save space)...")
for root, dirs, files in os.walk(raw_dataset):
    path_lower = root.lower()
    
    implicit_label = None
    if 'original' in path_lower or 'authentic' in path_lower or 'real' in path_lower:
        implicit_label = 'authentic'
    elif 'fake' in path_lower or 'spoof' in path_lower or 'ai' in path_lower:
        implicit_label = 'ai_generated'
        
    for f in files:
        if f.endswith(('.wav', '.mp3', '.flac')):
            fname_no_ext = f.replace('.wav', '').replace('.mp3', '').replace('.flac', '').lower()
            
            target_label = label_map.get(fname_no_ext, implicit_label)
                
            if target_label == 'authentic':
                # MOVE file instead of copy
                shutil.move(os.path.join(root, f), os.path.join(authentic_dir, f))
            elif target_label == 'ai_generated':
                shutil.move(os.path.join(root, f), os.path.join(ai_generated_dir, f))

# CLEANUP RAW DATASET TO FREE MORE SPACE
shutil.rmtree(raw_dataset, ignore_errors=True)

print("\n--- Routing Complete ---")
print(f"Total Authentic Files: {len(os.listdir(authentic_dir))}")
print(f"Total AI Generated Files: {len(os.listdir(ai_generated_dir))}")

### Step 2: GPU-Accelerated Preprocessing (ANTI-DISCONNECT)
We convert the audio to `.npy` formats using **TorchAudio** directly on the GPU. This is up to 10x faster than Librosa and keeps the GPU active so Colab doesn't kick you out for idling!

In [ ]:
import numpy as np
import torch
import torchaudio
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Preprocessing running on: {device}")

OUTPUT_PATH_NPY = "/content/dataset_npy"  
os.makedirs(OUTPUT_PATH_NPY, exist_ok=True)

N_MELS = 128
MAX_TIME_STEPS = 128
categories = ["authentic", "ai_generated"]

# Define torchaudio transforms on the GPU
mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_mels=N_MELS, n_fft=1024, hop_length=512).to(device)
amplitude_to_db = torchaudio.transforms.AmplitudeToDB().to(device)

for category in categories:
    in_dir = os.path.join(DATASET_AUDIO_PATH, category)
    out_dir = os.path.join(OUTPUT_PATH_NPY, category)
    os.makedirs(out_dir, exist_ok=True)
        
    files = [f for f in os.listdir(in_dir) if f.endswith(('.wav', '.mp3', '.flac'))]
    print(f"Processing {len(files)} files in {category} to Spectrograms...")
    
    for f in tqdm(files):
        out_filename = os.path.splitext(f)[0] + ".npy"
        out_path = os.path.join(out_dir, out_filename)
        if os.path.exists(out_path): continue
        try:
            audio_path = os.path.join(in_dir, f)
            
            # Load audio using torchaudio (much faster)
            waveform, sr = torchaudio.load(audio_path)
            if len(waveform.shape) > 1 and waveform.shape[0] > 1: 
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            # Resample if needed
            if sr != 16000: 
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                waveform = resampler(waveform)
                sr = 16000
            
            # Move to GPU for lightning-fast processing
            waveform = waveform.to(device)
            
            # Generate Mel Spectrogram on GPU
            S = mel_transform(waveform)
            S_dB = amplitude_to_db(S).squeeze(0)
            
            # Normalize
            S_normalized = (S_dB - S_dB.min()) / (S_dB.max() - S_dB.min() + 1e-9)
            
            # Pad or trim
            if S_normalized.shape[1] < MAX_TIME_STEPS:
                pad_width = MAX_TIME_STEPS - S_normalized.shape[1]
                S_normalized = torch.nn.functional.pad(S_normalized, (0, pad_width, 0, 0), mode='constant', value=0)
            else:
                S_normalized = S_normalized[:, :MAX_TIME_STEPS]
                
            # Move back to CPU and save
            np.save(out_path, S_normalized.cpu().numpy())
            
            # AGGRESSIVE DISK CLEANUP
            os.remove(audio_path)
        except Exception:
            pass
            
print("\nPreprocessing Complete!")

### Step 3: Train Neural Network
Trains your custom `AudioCNN` for 15 epochs, saving a checkpoint model to your personal Google Drive after every single epoch.

In [ ]:
import random
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(128 * 8 * 8, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = self.pool4(F.relu(self.conv4(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class AudioDataset(Dataset):
    def __init__(self, file_paths, labels, n_mels=128, max_time_steps=128):
        self.file_paths = file_paths
        self.labels = labels
        self.n_mels = n_mels
        self.max_time_steps = max_time_steps

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        try:
            S_normalized = np.load(file_path)
            img_tensor = torch.tensor(S_normalized, dtype=torch.float32).unsqueeze(0)
            label_tensor = torch.tensor([label], dtype=torch.float32)
            return img_tensor, label_tensor
        except Exception:
            return torch.zeros((1, self.n_mels, self.max_time_steps)), torch.tensor([label], dtype=torch.float32)

def prepare_data(dataset_path):
    authentic_dir = os.path.join(dataset_path, "authentic")
    ai_dir = os.path.join(dataset_path, "ai_generated")
    authentic_files = [os.path.join(authentic_dir, f) for f in os.listdir(authentic_dir) if f.endswith('.npy')] if os.path.exists(authentic_dir) else []
    ai_files = [os.path.join(ai_dir, f) for f in os.listdir(ai_dir) if f.endswith('.npy')] if os.path.exists(ai_dir) else []
    if len(authentic_files) == 0 and len(ai_files) == 0: return [], []
        
    # To avoid bias, we balance the dataset if there is a severe mismatch
    max_len = max(len(authentic_files), len(ai_files))
    if len(authentic_files) < max_len and len(authentic_files) > 0:
        authentic_files = (authentic_files * (max_len // len(authentic_files) + 1))[:max_len]
    if len(ai_files) < max_len and len(ai_files) > 0:
        ai_files = (ai_files * (max_len // len(ai_files) + 1))[:max_len]
        
    all_files = authentic_files + ai_files
    all_labels = [0] * len(authentic_files) + [1] * len(ai_files) # 0 for Authentic, 1 for AI Fake
    
    combined = list(zip(all_files, all_labels))
    random.shuffle(combined)
    all_files[:], all_labels[:] = zip(*combined)
    return all_files, all_labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nTraining on device: {device}")

DATASET_PATH_NPY = "/content/dataset_npy"
MODEL_SAVE_PATH = "/content/drive/MyDrive/audio_model.pth"
EPOCHS = 15
BATCH_SIZE = 32

model = AudioCNN().to(device)

# ======================================================================
# CHECK FOR EXISTING MODEL TO CONTINUE TRAINING (TRANSFER LEARNING)
# ======================================================================
if os.path.exists(MODEL_SAVE_PATH):
    print(f"\n>>> FOUND EXISTING MODEL: {MODEL_SAVE_PATH} <<<")
    print(">>> Loading weights to CONTINUE training (Fine-Tuning) instead of starting from scratch... <<<")
    try:
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
        print(">>> Successfully loaded previous model weights! <<<\n")
    except Exception as e:
        print(f">>> Error loading existing model: {e} <<<")
        print(">>> Starting from scratch instead. <<<\n")
else:
    print(f"\n>>> No existing model found at {MODEL_SAVE_PATH}. Starting training from scratch. <<<\n")

files, labels = prepare_data(DATASET_PATH_NPY)

if len(files) == 0:
    print("No preprocessed data found! Check earlier steps.")
else:
    split_idx = int(0.8 * len(files))
    train_loader = DataLoader(AudioDataset(files[:split_idx], labels[:split_idx]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(AudioDataset(files[split_idx:], labels[split_idx:]), batch_size=BATCH_SIZE, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("Starting training loop...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
            
        train_acc = 100. * correct / total
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                predicted = (torch.sigmoid(outputs) > 0.5).float()
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()
                
        val_acc = 100. * val_correct / val_total if val_total > 0 else 0
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}%")

        # Save a checkpoint to Google Drive after EVERY epoch
        epoch_save_path = MODEL_SAVE_PATH.replace('.pth', f'_epoch_{epoch+1}.pth')
        torch.save(model.state_dict(), epoch_save_path)
        print(f"Saved checkpoint: {epoch_save_path}")

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"\nTraining Complete! Your final model was saved directly to your Google Drive at: {MODEL_SAVE_PATH}")